<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/codebook/training_temp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.0 MB/s eta 0:00:00


In [3]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time
import numpy as np
import math
import sys

In [4]:
REPO_DIR = '/content/Texture_synthesis'
os.chdir('/content')

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/ValentinaEmili/Texture-synthesis.git Texture_synthesis

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

Cloning into 'Texture_synthesis'...
remote: Enumerating objects: 1024, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 1024 (delta 73), reused 9 (delta 9), pack-reused 858 (from 3)
Receiving objects: 100% (1024/1024), 170.34 MiB | 18.88 MiB/s, done.
Resolving deltas: 100% (442/442), done.


In [5]:
!pip install import-ipynb -q
import import_ipynb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 10.6 MB/s eta 0:00:00


In [6]:
#model_type = 'vq_vae'
#model_type = 'vq_gan'
model_type = 'vq_gan_tt'

split = '1'
#split = '2'
#split = '1_2'

#batch_size, size = 8, 256
batch_size, size = 4, 256

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if model_type == 'vq_vae':
  from Texture_synthesis.codebook.VQ_VAE import VQ_VAE

  generator = VQ_VAE(hidden_dim=128, embedding_dim=128, num_embeddings=512).to(device)
  optimizer_gen = optim.Adam(generator.parameters(), lr=5e-5, betas=(0.9, 0.999))

  save_folder = f'/content/drive/MyDrive/DeepLearning/dtd/new_checkpoints/{model_type}'
  gen_save_path = os.path.join(save_folder, f"best_{split}.pth")

elif model_type == 'vq_gan':
  from Texture_synthesis.codebook.VQGAN import VQGAN, Discriminator

  generator = VQGAN(hidden_dim=128, embedding_dim=128, num_embeddings=512).to(device)
  optimizer_gen = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.9))

  discriminator = Discriminator().to(device)
  optimizer_disc = optim.Adam(discriminator.parameters(), lr=5e-5, betas=(0.5, 0.9))

  save_folder = f'/content/drive/MyDrive/DeepLearning/dtd/new_checkpoints/{model_type}'
  gen_save_path = os.path.join(save_folder, f"gen_best_{split}.pth")
  disc_save_path = os.path.join(save_folder, f"disc_best_{split}.pth")

else:
  from Texture_synthesis.codebook.VQGAN_TT import VQGAN, Discriminator

  generator = VQGAN(num_embeddings=1024).to(device)
  optimizer_gen = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.9))

  discriminator = Discriminator().to(device)
  optimizer_disc = optim.Adam(discriminator.parameters(), lr=5e-5, betas=(0.5, 0.9))

  save_folder = '/content/drive/MyDrive/DeepLearning/dtd/new_checkpoints/vq_gan/taming_transformers'
  os.makedirs(save_folder, exist_ok=True)
  gen_save_path = os.path.join(save_folder, f"gen_best_{split}.pth")
  disc_save_path = os.path.join(save_folder, f"disc_best_{split}.pth")

perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device)
perceptual_loss_fn.eval()
for p in perceptual_loss_fn.parameters():
    p.requires_grad_(False)

train_file = 'train_' + split + '.txt'
val_file = 'val_' + split + '.txt'
test_file = 'test_' + split + '.txt'

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 129MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/vgg.pth


VQ-VAE vs  VQGAN to verify the effect of adding perceptual and adversarial objectives to the VQ-VAE

* hidden_dim=128, embedding_dim=128, num_embeddings=512
* hidden_dim=128, embedding_dim=256, num_embeddings=512

Let's test Taming VQGAN with num_embeddings$\in${1024, 512, 256} and embedding_dim=256

In [8]:
train_transform = transforms.Compose([
        transforms.Resize(300),
        transforms.RandomCrop(size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(300),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [9]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"

train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, train_file), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, val_file), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, test_file), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

## Validation

In [10]:
def visual_validation(model, val_loader, device, show_imgs=True):
  model.eval()

  all_indices = []
  visual_samples = None
  total_loss, total_percept, total_recon, total_vq = 0.0, 0.0, 0.0, 0.0

  num_embeddings = model.vq.num_embeddings
  embedding_dim = model.vq.embedding_dim
  embeddings = model.vq.embeddings.weight

  with torch.no_grad():
    for batch_idx, (data, _) in enumerate(val_loader):
      data = data.to(device)
      data_recon, vq_loss = model(data)

      visual_samples = (data.cpu(), data_recon.cpu())

      if model_type == 'vq_vae':
        metric_data = F.interpolate(data, size=(256, 256), mode='bilinear', align_corners=False)
        metric_data_recon = F.interpolate(data_recon, size=(256, 256), mode='bilinear', align_corners=False)

        recon_loss = F.mse_loss(metric_data_recon, metric_data)
        percept_loss = perceptual_loss_fn(metric_data_recon, metric_data).mean()
        loss = percept_loss + vq_loss + recon_loss * 0.2

      else:
        recon_loss = F.mse_loss(data_recon, data)
        percept_loss = perceptual_loss_fn(data_recon, data).mean()
        loss = percept_loss + vq_loss + recon_loss * 0.2

      total_recon += recon_loss.item()
      total_percept += percept_loss.item()
      total_vq += vq_loss.item()
      total_loss += loss.item()

      z = model.encoder(data)
      if model_type == 'vq_gan_tt': z = model.quant_conv(z)
      z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)

      distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))

      indices = torch.argmin(distances, dim=1)
      all_indices.append(indices.cpu())

  # percentage of used vectors
  encoding_indices = torch.cat(all_indices)
  unique_indices = torch.unique(encoding_indices)
  util_percen = len(unique_indices) / num_embeddings * 100

  # perplexity
  counts = torch.bincount(encoding_indices, minlength=num_embeddings).float()
  probs = counts / counts.sum()
  perplexity = torch.exp(-torch.sum(probs * torch.log(probs + 1e-10)))

  avg_loss = total_loss / len(val_loader)
  avg_recon = total_recon / len(val_loader)
  avg_percept = total_percept / len(val_loader)
  avg_vq = total_vq / len(val_loader)

  selection_score = avg_percept + 0.2 * avg_recon
  unique_vectors_used = (len(unique_indices) / num_embeddings) * 100

  # visual reconstruction plotting
  if (show_imgs):
    print(f"Perceptual Loss: {avg_percept:.4f}  \nReconstruction Loss: {avg_recon:.4f}  \nCodebook Loss: {avg_vq:.4f}")
    print(f"Unique Vectors Used: {unique_vectors_used:.4f}%  \nCodebook Utilization: {util_percen:.2f}%  \nCodebook Perplexity: {perplexity.item():.2f}")

    real_imgs, recon_imgs = visual_samples
    num_displayed_imgs = min(8, real_imgs.shape[0])

    fig, axes = plt.subplots(2, num_displayed_imgs, figsize=(num_displayed_imgs * 3, 6))

    for i in range(num_displayed_imgs):
      real_img = real_imgs[i].permute(1, 2, 0).numpy()
      recon_img = recon_imgs[i].permute(1, 2, 0).numpy()

      real_plot = ((real_img + 1) / 2).clip(0, 1)
      recon_plot = ((recon_img + 1) / 2).clip(0, 1)

      axes[0, i].imshow(real_plot)
      axes[0, i].set_title(f"original {i+1}")
      axes[0, i].axis('off')

      axes[1, i].imshow(recon_plot)
      axes[1, i].set_title(f"reconstructed {i+1}")
      axes[1, i].axis('off')

    plt.show()

  else:
    print(f"Val Loss: {avg_loss:.4f}  | Perceptual Loss: {avg_percept:.4f}  | Reconstruction Loss: {avg_recon:.4f}  | Codebook Loss: {avg_vq:.4f}")
    print(f"Unique Vectors Used: {unique_vectors_used:.4f}%  | Codebook Utilization: {util_percen:.2f}%  | Codebook Perplexity: {perplexity.item():.2f}")

  return selection_score, unique_vectors_used, perplexity.item()

## Training

We compute the adaptive weight $λ$ according to:

$\lambda = \frac{\nabla _{G_{L}} [L_{recon} + L_{percep}]}{\nabla _{G_{L}} [L_{GAN} + ϵ]}$

where the $L_{recon}$ is the perceptual reconstruction loss $\nabla _{G_{L}}[\cdot]$ is the gradient of its input w.r.t. the last layer $L$ of the decoder.

In [11]:
def calculate_adaptive_weight(recon_loss, g_loss, last_layer_weights):
    recon_grads = torch.autograd.grad(recon_loss, last_layer_weights, retain_graph=True, allow_unused=True)[0]
    g_grads = torch.autograd.grad(g_loss, last_layer_weights, retain_graph=True, allow_unused=True)[0]

    if recon_grads is None:
      recon_norm = torch.tensor(0., device=last_layer_weights.device)
    else:
      recon_norm = torch.norm(recon_grads)

    if g_grads is None:
      g_norm = torch.tensor(0., device=last_layer_weights.device)
    else:
      g_norm = torch.norm(g_grads)

    lambda_weight = recon_norm / (g_norm + 1e-4)
    lambda_weight = torch.clamp(lambda_weight, 0.0, 1e4).detach()
    return lambda_weight

During the Generator training we freeze the Discriminator so that calling `backward()` on the Generator's loss we do not update the Discriminator's weights.

`disc_start_step`: before that, we only train the Generator so like a standard VQ-VAE. From that step onwards, the Discriminator is introduced into the training process.

In [12]:
def set_requires_grad(model, requires_grad):
  for param in model.parameters():
    param.requires_grad = requires_grad

def train_step(generator, discriminator, perceptual_loss_fn, data, optimizer_g, optimizer_d, disc_start_step, current_global_step):
    # GENERATOR

    # disable discriminator gradients while training generator
    set_requires_grad(discriminator, False)

    optimizer_g.zero_grad()
    recon_batch, vq_loss = generator(data)
    recon_loss = F.mse_loss(recon_batch, data)
    percept_loss = perceptual_loss_fn(recon_batch, data).mean()

    # dynamic adaptive weight scheduling
    if current_global_step >= disc_start_step:
        fake_outputs = discriminator(recon_batch)
        g_loss = -fake_outputs.mean()   # hinge loss

        if model_type == 'vq_gan': last_layer_weights = generator.decoder.block5.weight
        else: last_layer_weights = generator.decoder.conv_out.weight
        disc_weight = calculate_adaptive_weight(recon_loss + percept_loss, g_loss, last_layer_weights)

        gen_loss = vq_loss + recon_loss + percept_loss + (disc_weight * g_loss)
    else:
        gen_loss = vq_loss + recon_loss + percept_loss

    gen_loss.backward()
    optimizer_g.step()
    set_requires_grad(discriminator, True)

    # DISCRIMINATOR
    disc_loss = torch.tensor(0.0, device=data.device)
    if current_global_step >= disc_start_step:
        optimizer_d.zero_grad()

        real_outputs = discriminator(data)
        fake_outputs_d = discriminator(recon_batch.detach())

        loss_real = F.relu(1.0 - real_outputs).mean()
        loss_fake = F.relu(1.0 + fake_outputs_d).mean()

        disc_loss = 0.5 * loss_real + 0.5 * loss_fake
        disc_loss.backward()
        optimizer_d.step()

    return gen_loss.item(), disc_loss.item()

In [13]:
def train_and_validation(
    generator, optimizer_gen,
    train_loader, val_loader, device, gen_save_path,
    discriminator=None, optimizer_disc=None, disc_save_path=None,
    best_val_score=float('inf'), disc_start_step=0, global_step=0, next_epoch=0, epochs=20, show_imgs=True):

  for epoch in range(next_epoch, epochs):

    generator.train()

    # VQGAN training
    if discriminator:
      discriminator.train()
      total_loss_gen, total_loss_disc = 0.0, 0.0

      for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)

        gen_loss, disc_loss = train_step(
                                    generator, discriminator, perceptual_loss_fn, data,
                                    optimizer_gen, optimizer_disc,
                                    disc_start_step, global_step
                                    )

        global_step += 1

        total_loss_gen += gen_loss
        total_loss_disc += disc_loss

      avg_gen_loss = total_loss_gen / len(train_loader)
      avg_disc_loss = total_loss_disc / len(train_loader)

      print(f"====> Epoch {epoch} Finished | Avg G-Loss: {avg_gen_loss:.4f} | Avg D-Loss: {avg_disc_loss:.4f}")

      val_score, unique_vectors_used, perplexity = visual_validation(generator, val_loader, device, show_imgs=show_imgs)

      if val_score < best_val_score:
        best_val_score = val_score
        print(f"New best epoch is epoch {epoch}\n")

        torch.save({
            'epoch': epoch,
            'model_state_dict': generator.state_dict(),
            'optimizer_state_dict': optimizer_gen.state_dict(),
            'best_val_score': val_score,
            'unique_vectos_used_percentage': unique_vectors_used,
            'perplexity': perplexity
        }, gen_save_path)

        torch.save({
            'epoch': epoch,
            'model_state_dict': discriminator.state_dict(),
            'optimizer_state_dict': optimizer_disc.state_dict(),
            'global_step': global_step
        }, disc_save_path)

    # VQ-VAE training
    else:
      total_loss, total_percept, total_recon, total_vq = 0.0, 0.0, 0.0, 0.0

      for batch_idx, (data, _) in enumerate(train_loader):
          data = data.to(device)
          optimizer_gen.zero_grad()
          data_recon, vq_loss = generator(data)                         # codebook loss

          recon_loss = F.mse_loss(data_recon, data)                                           # reconstruction loss
          percept_loss = perceptual_loss_fn(data_recon, data).mean()                          # perceptual loss
          loss = percept_loss + vq_loss + recon_loss * 0.2
          loss.backward()

          optimizer_gen.step()

          total_loss += loss.item()

          total_percept += percept_loss.item()
          total_recon += recon_loss.item()
          total_vq += vq_loss.item()

      train_loss = total_loss / len(train_loader)
      train_percept = total_percept / len(train_loader)
      train_recon = total_recon / len(train_loader)
      train_vq = total_vq / len(train_loader)

      print(f"====> Train Loss: {train_loss:.4f}  | Perceptual Loss: {train_percept:.4f}  | Reconstruction Loss: {train_recon:.4f}  | Codebook Loss: {train_vq:.4f}\n")

      val_score, unique_vectors_used, perplexity = visual_validation(generator, val_loader, device, show_imgs=False)

    if val_score < best_val_score:
      best_val_score = val_score
      print(f"New best epoch is epoch {epoch}\n")

      torch.save({
          'epoch': epoch,
          'model_state_dict': generator.state_dict(),
          'optimizer_state_dict': optimizer_gen.state_dict(),
          'best_val_score': val_score,
          'unique_vectors_used_percentage': unique_vectors_used * 100,
          'perplexity': perplexity
      }, gen_save_path)

In [15]:
if model_type == 'vq_vae':
  train_and_validation(
      generator, optimizer_gen,
      train_loader, val_loader, device, gen_save_path,
      epochs=40, show_imgs=False)

elif model_type == 'vq_gan':
  train_and_validation(
      generator, optimizer_gen,
      train_loader, val_loader, device, gen_save_path,
      discriminator=discriminator, optimizer_disc=optimizer_disc, disc_save_path=disc_save_path,
      disc_start_step=800, epochs=40, show_imgs=False)

else:
  train_and_validation(
      generator, optimizer_gen,
      train_loader, val_loader, device, gen_save_path,
      discriminator=discriminator, optimizer_disc=optimizer_disc, disc_save_path=disc_save_path,
      disc_start_step=1000, epochs=40, show_imgs=False)

====> Epoch 0 Finished | Avg G-Loss: 1.2244 | Avg D-Loss: 0.0000
Val Loss: 0.8003  | Perceptual Loss: 0.6870  | Reconstruction Loss: 0.1599  | Codebook Loss: 0.0813
Unique Vectors Used: 1.7578%  | Codebook Utilization: 1.76%  | Codebook Perplexity: 2.83
New best epoch is epoch 0

====> Epoch 1 Finished | Avg G-Loss: 1.0505 | Avg D-Loss: 0.0000
Val Loss: 0.7807  | Perceptual Loss: 0.6490  | Reconstruction Loss: 0.1342  | Codebook Loss: 0.1049
Unique Vectors Used: 1.6602%  | Codebook Utilization: 1.66%  | Codebook Perplexity: 5.75
New best epoch is epoch 1



AttributeError: 'Decoder' object has no attribute 'block5'

In [ ]:
checkpoint = torch.load(gen_save_path, weights_only=True)
generator.load_state_dict(checkpoint['model_state_dict'])
optimizer_gen.load_state_dict(checkpoint['optimizer_state_dict'])
best_val_score = checkpoint['best_val_score']
next_epoch = checkpoint['epoch'] + 1

checkpoint = torch.load(disc_save_path, weights_only=True)
discriminator.load_state_dict(checkpoint['model_state_dict'])
optimizer_disc.load_state_dict(checkpoint['optimizer_state_dict'])
global_step = checkpoint['global_step']

if model_type == 'vq_vae':
  train_and_validation(
      generator, optimizer_gen,
      train_loader, val_loader, device, gen_save_path,
      epochs=40, show_imgs=False)

elif model_type == 'vq_gan':
  train_and_validation(
      generator, optimizer_gen,
      train_loader, val_loader, device, gen_save_path,
      discriminator=discriminator, optimizer_disc=optimizer_disc, disc_save_path=disc_save_path,
      disc_start_step=800, epochs=40, show_imgs=False)

else:
  train_and_validation(
      generator, optimizer_gen,
      train_loader, val_loader, device, gen_save_path,
      discriminator=discriminator, optimizer_disc=optimizer_disc, disc_save_path=disc_save_path,
      disc_start_step=1000, epochs=40, show_imgs=False,
      best_val_score=best_val_score, next_epoch=next_epoch, global_step=global_step)

====> Epoch 2 Finished | Avg G-Loss: 0.7051 | Avg D-Loss: 0.8739
Val Loss: 0.8094  | Perceptual Loss: 0.7180  | Reconstruction Loss: 0.2184  | Codebook Loss: 0.0477
Unique Vectors Used: 1.4648%  | Codebook Utilization: 1.46%  | Codebook Perplexity: 6.55
====> Epoch 3 Finished | Avg G-Loss: 0.4837 | Avg D-Loss: 0.9992
Val Loss: 0.9149  | Perceptual Loss: 0.8095  | Reconstruction Loss: 0.2843  | Codebook Loss: 0.0485
Unique Vectors Used: 1.7578%  | Codebook Utilization: 1.76%  | Codebook Perplexity: 8.05
====> Epoch 4 Finished | Avg G-Loss: 0.4945 | Avg D-Loss: 1.0024
Val Loss: 0.8152  | Perceptual Loss: 0.7339  | Reconstruction Loss: 0.3162  | Codebook Loss: 0.0180
Unique Vectors Used: 1.6602%  | Codebook Utilization: 1.66%  | Codebook Perplexity: 11.15
====> Epoch 5 Finished | Avg G-Loss: 0.3205 | Avg D-Loss: 1.0038
Val Loss: 0.7954  | Perceptual Loss: 0.7339  | Reconstruction Loss: 0.2393  | Codebook Loss: 0.0137
Unique Vectors Used: 2.0508%  | Codebook Utilization: 2.05%  | Codebook 

VQ-VAE:
* split 1:
  * embedding_dim=128:

    Val Loss: 0.7394  | Perceptual Loss: 0.5283  | Reconstruction Loss: 0.0886  | Codebook Loss: 0.1934 | Unique Vectors Used: 36.7188%  | Codebook Utilization: 36.72%  | Codebook Perplexity: 43.11

  * embedding_dim=256:

    Val Loss: 0.7248  | Perceptual Loss: 0.5411  | Reconstruction Loss: 0.1304  | Codebook Loss: 0.1576 | Unique Vectors Used: 11.1328%  | Codebook Utilization: 11.13%  | Codebook Perplexity: 28.90

VQGAN:
* split 1:
  * embedding_dim=128:

    Avg G-Loss: 0.9552 | Avg D-Loss: 0.9237
    
    Val Loss: 0.7606  | Perceptual Loss: 0.5734  | Reconstruction Loss: 0.1112  | Codebook Loss: 0.1649 | Unique Vectors Used: 14.6484%  | Codebook Utilization: 14.65%  | Codebook Perplexity: 41.41